In [ ]:
import anndata as ad
import math
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.stats as stats
import seaborn as sns
import statsmodels.api as sm
import warnings
import os  
import loompy
import gzip
import shutil
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests

%matplotlib inline
%config InlineBackend.figure_format = 'retina'


sc.settings.verbosity = 2
sc.settings.autoshow = False
sc.settings.set_figure_params(dpi=50, dpi_save=300, format='png', 
                             frameon=False, transparent=True, fontsize=10, figsize=(4, 4))

warnings.simplefilter(action='ignore', category=FutureWarning)

plt.rcParams["image.aspect"] = "equal"
plt.rcParams["figure.figsize"] = ([4, 4])  
mpl.rcParams['pdf.fonttype'] = 42

colorrs = ["#4DBBD5", "#00A087", "#E64B35","#3C5488", "#F39B7F", "#8491B4",
        "#91D1C2",  "#B9C984",  "#9ACBDE", "#F494BE", "#EDCAE0", 
        "#C8CADF", "#F47892", "#F6A395",  "#C9AFA2", "#ABADC5", "#AEB9AC", 
        "#4b6aa8", "#3ca0cf", "#c376a7", "#ad98c3", "#cea5c7",
        "#53738c", "#a5a9b0", "#a78982", "#696a6c", "#92699e",
        "#d69971", "#df5734", "#6c408e", "#ac6894", "#d4c2db",
        "#537eb7", "#83ab8e", "#ece399", "#405993", "#cc7f73",
        "#b95055", "#d5bb72", "#bc9a7f", "#e0cfda", "#d8a0c0",
        "#d69a55", "#64a776", "#cbdaa9",
        "#efd2c9", "#da6f6d", "#ebb1a4", "#a44e89", "#a9c2cb",
        "#b85292", "#6d6fa0", "#8d689d", "#c8c7e1", "#d25774",
        "#c49abc", "#927c9a", "#3674a2", "#9f8d89", "#72567a",
        "#63a3b8", "#c4daec", "#61bada", "#b7deea", "#e29eaf",
        "#4490c4", "#e6e2a3",  "#c4612f", "#9a70a8",
        "#76a2be", "#408444", "#c6adb0", "#9d3b62", "#2d3462"]

In [ ]:
sample_info_path = "/home/xiaoquan/scanpy/KP/CellRanger/sample.csv"
sample_info = pd.read_csv(sample_info_path)

sample_dirs = [
    "M01", "M02", "M03", "M04", "M05", "M06", "M07", "M08", "M09", "M10",
    "M11", "M12", "M13", "M14", "M15", "M16", "M17", "M18", "M19", "M20",
    "M21", "M22", "M23", "S01", "S02", "S03", "S04", "S05", "S06", "S07",
    "S08", "S09", "S10", "S11", "S12", "S13", "S14", "S15", "S16", "S17",
    "S18", "S19", "S20", "S21", "S22", "S23", "HC01", "HC02", "HC03", "HC04",
    "HC05", "HC06", "HC07", "HC08", "HC09", "HC10", "HC11", "HC12", "HC13", "HC14",
    "HC15", "HC16", "HC17", "HC18", "HC19", "HC20", "HC21", "HC22", "HC23", "HC24",
    "HC25", "HC26", "HC27", "HC28", "HC29", "HC30", "HC31", "HC32", "HC33", "HC34",
    "HC35", "HC36", "HC37", "HC38", "HC39", "HC40", "HC41", "HC42", "HC43", "HC44",
    "HC45", "HC46", "HC47", "HC48", "HC49", "HC50", "HC51", "HC52", "HC53", "HC54"
]

base_path = "/home/xiaoquan/scanpy/KP/CellRanger"
adatas = []
for sd in sample_dirs:
    path = os.path.join(base_path, sd)
    ad = sc.read_10x_mtx(path, var_names='gene_symbols')
    ad.obs["Sample"] = sd
    if sd.startswith("HC"):
        group = "Healthy Controls"
        condition = "HC"
    elif sd.startswith("M"):
        group = "Mild Klebsiella pneumoniae pneumonia" 
        condition = "MKPP"
    else:
        group = "Severe Klebsiella pneumoniae pneumonia"
        condition = "SKPP"
    
    ad.obs["Group"] = group
    ad.obs["Condition"] = condition 
    adatas.append(ad)

adata = sc.concat(adatas, label='Sample',keys=sample_dirs, index_unique='-',join="inner")

In [ ]:
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

In [ ]:
adata

In [ ]:
import scrublet as scr

In [ ]:
sc.external.pp.scrublet(adata, expected_doublet_rate=0.05, threshold=0.25, batch_key="Sample")

In [ ]:
adata = adata[adata.obs["predicted_doublet"] == False]

In [ ]:
def basic_qc(adata):

    adata.var['mt'] = adata.var_names.str.startswith('MT-')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    mt_gene = adata.var.index[adata.var_names.str.startswith('MT-')]
    
    adata.var["mt"] = adata.var_names.str.startswith("MT-")
    adata.var['rp'] = [i.startswith('RPL') or i.startswith('RPS') for i in adata.var_names]
    sc.pp.calculate_qc_metrics(adata, qc_vars=['rp'], percent_top=None, log1p=False, inplace=True)
    rp_gene = [i for i in adata.var_names if i.startswith('RPL') or i.startswith('RPS')]
    
    adata.var['hb'] = [i.startswith('HB') and not i.startswith('HBP') for i in adata.var_names]
    sc.pp.calculate_qc_metrics(adata, qc_vars=['hb'], percent_top=None, log1p=False, inplace=True)
    hb_gene = [i for i in adata.var_names if i.startswith('HB') and not i.startswith('HBP')]
     adata = adata[adata.obs.n_genes_by_counts < 5000, :]
    adata = adata[adata.obs.n_genes_by_counts > 200, :] 
    adata = adata[adata.obs.pct_counts_mt < 10, :]
    adata = adata[adata.obs.pct_counts_rp > 3, :]
    adata = adata[adata.obs.pct_counts_hb < 1, :]   
    return adata

In [ ]:
adata = basic_qc(adata)

In [ ]:
import harmonypy as hm 

In [ ]:
adata.layers["counts"] = adata.X.copy()

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
adata.raw = adata
adata.layers["log1p_norm"] = adata.X.copy()

In [ ]:
adata.var["mt"] = adata.var_names.str.startswith("MT-")
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
adata.var["ig"] = adata.var_names.str.startswith(("IGH", "IGK", "IGL"))

In [ ]:
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5, n_top_genes=1500)
adata.var["hvg_original"] = adata.var["highly_variable"].copy()
adata.var["hvg_for_pca"] = (
    adata.var["highly_variable"]
    & ~adata.var["mt"]
    & ~adata.var["rp"]
    & ~adata.var["ig"]
)
adata.var["highly_variable"] = adata.var["hvg_for_pca"].copy()

In [ ]:
adata.X = adata.layers["log1p_norm"].copy()
sc.pp.regress_out(adata, ['total_counts', 'pct_counts_mt'])
sc.pp.scale(adata, max_value=10)
sc.tl.pca(
    adata,
    svd_solver="arpack",
    use_highly_variable=True
)

In [ ]:
sc.external.pp.harmony_integrate(adata, 
                                key=['Sample', 'Dataset'],  
                                theta=[2.5, 1.5], basis='X_pca', adjusted_basis='X_pca_harmony')  

In [ ]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=20, use_rep='X_pca_harmony')

In [ ]:
sc.tl.umap(adata)
sc.pl.umap(adata, color='Sample', palette=colorrs, title='After Harmony', legend_loc=None)

In [ ]:
resolutions = [0.5, 1.0, 1.5, 2.0]
for res in resolutions:
    sc.tl.louvain(adata, resolution=res, key_added=f'louvain_{res}')

In [ ]:
for k in ["louvain_0.5", "louvain_1.0", "louvain_1.5", "louvain_2.0", "Group"]:
    sc.pl.umap(adata,color=k, palette=colorr)

In [ ]:
sc.tl.rank_genes_groups(
    T_cell,
    groupby="louvain_2.0",
    method="wilcoxon",
    use_raw=True,
    pts=True,
    key_added="rank_genes_groups"
)

In [ ]:
markers = ["CD79A", "CD79B","MS4A1",
           "MZB1",  "IGHG1", 
           "CD3D", "CD3E", "CD40LG", 
           "CD8A", "CD8B",
           "TRDV2", "TRGV9", 
           "SLC4A10", "TRAV1-2", 
           "KLRF1", "NKG7", "TYROBP", 
          "CST3", "LYZ", "CD14","FCGR3A","CD1C", "ITGAX",
          "PPBP", "PF4"]
sc.pl.dotplot(adata, var_names=markers, groupby='louvain_2.0', ax=ax, 
              standard_scale='var', use_raw=True, show=False)

In [ ]:
condition_order = ["HC", "MKPP", "SKPP"]
condition_labels = ["Healthy controls", "Mild symptoms", "Severe symptoms"]
condition_colors = {
    "HC": "#4DBBD5B2", 
    "MKPP": "#00A087B2",  
    "SKPP": "#E64B35B2"
}

x_min, x_max = np.min(adata.obsm['X_umap'][:, 0]), np.max(adata.obsm['X_umap'][:, 0])
y_min, y_max = np.min(adata.obsm['X_umap'][:, 1]), np.max(adata.obsm['X_umap'][:, 1])

fig, axes = plt.subplots(1, len(condition_order), figsize=(3 * len(condition_order), 3.5), 
                        dpi=300, sharex=True, sharey=True,
                        gridspec_kw={'wspace': 0, 'hspace': 0})  

for i, condition in enumerate(condition_order):
    ax = axes[i]

    adata_bg = adata[adata.obs['Condition'] != condition]
    ax.scatter(
        adata_bg.obsm['X_umap'][:, 0],
        adata_bg.obsm['X_umap'][:, 1],
        s=0.01,
        color="lightgray",
        rasterized=True
    )

    adata_fg = adata[adata.obs['Condition'] == condition]
    ax.scatter(
        adata_fg.obsm['X_umap'][:, 0],
        adata_fg.obsm['X_umap'][:, 1],
        s=0.005,
        alpha=0.7,
        color=condition_colors[condition],
        rasterized=True
    )
    
    ax.set_title(condition_labels[i], fontsize=13, fontweight='bold')
    ax.set_facecolor("white")
    ax.grid(False)
    
    if i == 0: 
        ax.spines['left'].set_visible(True)
        ax.spines['right'].set_visible(False)
        ax.spines['top'].set_visible(False)
        ax.spines['bottom'].set_visible(True)
        ax.tick_params(left=True, labelleft=True, right=False)
    elif i == len(condition_order) - 1: 
        ax.spines['left'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['top'].set_visible(False)
        ax.spines['bottom'].set_visible(True)
        ax.tick_params(left=False, right=False, labelright=False)
    else: 
        ax.spines['left'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['top'].set_visible(False)
        ax.spines['bottom'].set_visible(True)
        ax.tick_params(left=False, right=False)

    ax.tick_params(bottom=False, labelbottom=False)

for ax in axes:
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

axes[0].set_ylabel('UMAP2', fontsize=12)
axes[1].set_xlabel('UMAP1', fontsize=12)

plt.show()

In [ ]:
count_df = adata.obs.groupby(["Condition", "celltype_major"]).size().reset_index(name="count")
count_df["Proportion"] = count_df.groupby(["Condition"])["count"].transform(lambda x: x/x.sum())

celltype_order = ['B', 'Plasma', 'CD4T', 'CD8T', 'γδT', 'MAIT', 'DNT', 'NK', 'Mono', 'DC', 'Mega']
count_df['celltype_major'] = pd.Categorical(count_df['celltype_major'], categories=celltype_order, ordered=True)
count_df = count_df.sort_values(['Condition', 'celltype_major'])

conditions = count_df['Condition'].unique()
n_groups = len(conditions)
colors = dict(zip(celltype_order, colorrs)) 

fig, axes = plt.subplots(1, n_groups, figsize=(n_groups * 3, 3), subplot_kw={'projection': 'polar'})

if n_groups == 1:
    axes = [axes]

for ax, cond in zip(axes, conditions):
    data = count_df[count_df['Condition'] == cond]

    ratios = data['Proportion'].values
    angles = np.concatenate(([0], np.cumsum(ratios) * 2 * np.pi))
    
     inner_radius = 1.5 
    outer_radius = 5 
    width = outer_radius - inner_radius
    
    for i, celltype in enumerate(data['celltype_major']):
        start_angle = angles[i]
        end_angle = angles[i+1]
        
        ax.bar(x=(start_angle + end_angle) / 2, 
               height=width, 
               width=end_angle - start_angle, 
               bottom=inner_radius,
               color=colors[celltype],
               edgecolor='white', 
               linewidth=1.0,
               alpha=1.0)


    ax.set_axis_off()
    ax.set_title(cond, fontsize=14, fontweight='bold', pad=5) 
plt.subplots_adjust(wspace=0.1)
handles = [plt.Rectangle((0,0),1,1, color=colors[ct]) for ct in celltype_order if ct in count_df['celltype_major'].unique()]
labels = [ct for ct in celltype_order if ct in count_df['celltype_major'].unique()]
fig.legend(handles, labels, title='Celltypes', bbox_to_anchor=(1.0, 0.5), loc='center left', fontsize=12, title_fontsize=14)

plt.show()